# Linear Algebra from Scratch
### This module covers foundational and important linear algebra tools dedicated for machine learning. 

It is not targetted to be 'economic' or 'competitive' but rather for a clear grasp of what happens inside the hood of modern cutting edge algortihms. With the philosophy that things only make sense when you understand their essence has lead me to work on this project.

Linear algebra certainly serves as a stepping stone for the evolution and advancements of modern artificial intelligence and deep learing. Therefore it is really important that we take account of this important field of mathematics for further poression in AI.

This notebook walks through the module chapter by chapter:

1. **Vectors** - arrows, lengths, and angles
2. **Matrices** - vectors that travel in packs, and the transformations they encode
3. **Determinants & Inverses** - measuring and undoing transformations
4. **QR Decomposition** - building perfectly perpendicular worlds (Gram-Schmidt)
5. **Eigenvalues & Eigenvectors** - the directions a transformation refuses to bend
6. **SVD, PCA & Pseudoinverse** - the grand finale, where all of the above assembles into a tool real machine learning systems depend on

Each chapter pairs a plain-English explanation, the underlying math notation and a runnable example against the module's actual code.

In [10]:
import sys
sys.path.insert(0, '.')  # so the module's files can be imported

from vector import Vector
from matrix import Matrix
from tools import svd, pinv, pca
import math, random
random.seed(42)

print("Module loaded. Let's begin the story.")


Module loaded. Let's begin the story.



---
## Chapter 1 — Vectors: an arrow with a memory

A **vector** is just an ordered list of numbers, but the moment you place it
in space it becomes an arrow: a direction and a distance rolled into one
object.

$$
\mathbf{v} = (v_1, v_2, \dots, v_n) \in \mathbb{R}^n
$$

In the module, this is the `Vector` class, a thin wrapper around a Python
list that knows how to add, subtract, scale, and measure itself.

### Length (the norm) :-

The most natural question to ask an arrow is *"how long are you?"* The
Euclidean length ($L^2$ norm) answers that with the Pythagorean theorem
generalized to $n$ dimensions:

$$
\lVert \mathbf{v} \rVert_2 = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}
$$

More generally, the module supports any generalised $L^p$ norm:

$$
\lVert \mathbf{v} \rVert_p = \left( \sum_{i=1}^n |v_i|^p \right)^{1/p}
$$


In [11]:
v = Vector([3, 4])
print("v =", v)
print("‖v‖₂ (Euclidean length) =", v.norm())      # classic 3-4-5 triangle
print("‖v‖₁ (Manhattan length) =", v.norm(1))

v = Vector([3.0, 4.0])
‖v‖₂ (Euclidean length) = 5.0
‖v‖₁ (Manhattan length) = 7.0


### Normalising — shrinking an arrow to a pure direction :-

Dividing a vector by its own length strips away the "how far" and keeps only
the "which way":

$$
\hat{\mathbf{v}} = \frac{\mathbf{v}}{\lVert \mathbf{v} \rVert}, \qquad \lVert \hat{\mathbf{v}} \rVert = 1
$$

In [12]:
unit_v = v.normalise()
print("unit vector:", unit_v)
print("its length:", unit_v.norm())

unit vector: Vector([0.6, 0.8])
its length: 1.0


### The dot product — measuring agreement between two arrows :-

$$
\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^n a_i b_i = \lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert \cos\theta
$$

When two vectors are perpendicular, $\theta = 90^\circ$ and $\cos\theta = 0$,
so their dot product vanishes. That is, a dot product of zero
meaning *orthogonality* is the seed that grows into QR decomposition and
eigendecomposition later in this story.

In [13]:
a = Vector([1, 0])
b = Vector([0, 1])

print("a · b =", a.dot(b))
print("orthogonal?", a.is_orthogonal_to(b))

c = Vector([1, 1])
print("a · c =", a.dot(c), "-> orthogonal?", a.is_orthogonal_to(c))

a · b = 0.0
orthogonal? True
a · c = 1.0 -> orthogonal? False


---
## Chapter 2 — Matrices: vectors travelling in formation 

A **matrix** is a rectangular grid of numbers but the more powerful way to
see it is as a *function* that transforms space: feed it a vector and it hands
back a new, transformed vector.

$$
A =
\begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n}\\
a_{21} & a_{22} & \cdots & a_{2n}\\
\vdots & \vdots & \ddots & \vdots\\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix}
\in \mathbb{R}^{m \times n}
$$

In the module, `Matrix` is built from a list of `Vector` rows, so everything
the `Vector` class already knows how to do (add, scale, dot product) is
reused rather than reimplemented.

In [14]:
M = Matrix([[1, 2], [3, 4], [5, 6]])
print(M)
print("shape:", M.shape)         
print("transpose:\n", M.transpose)


Matrix([
 Vector([1.0, 2.0]),
 Vector([3.0, 4.0]),
 Vector([5.0, 6.0])
])
shape: (3, 2)
transpose:
 Matrix([
 Vector([1.0, 3.0, 5.0]),
 Vector([2.0, 4.0, 6.0])
])


### Matrix multiplication — composing transformations :-

$$
(AB)_{ij} = \sum_{k} A_{ik} B_{kj}
$$

This is really just: *"take the dot product of row $i$ of $A$ with column
$j$ of $B$."* Two transformations chained into one.


In [15]:
A = Matrix([[2, 0], [0, 3]])   # stretches x by 2, y by 3
B = Matrix([[1, 1], [0, 1]])   # shearing

combined = A @ B
print("A @ B =\n", combined)

# apply the combined transformation to a column vector [1, 1]

x_col = Matrix([[1], [1]])
result = combined @ x_col
print("\ntransform the vector (1, 1) through the combined map:")
print(result)

A @ B =
 Matrix([
 Vector([2.0, 2.0]),
 Vector([0.0, 3.0])
])

transform the vector (1, 1) through the combined map:
Matrix([
 Vector([4.0]),
 Vector([3.0])
])



---
## Chapter 3 — Determinants and Inverses: measuring and undoing

### The determinant - how much a transformation stretches (or flattens) space :-

For a $2\times 2$ matrix:

$$
\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc
$$

For larger matrices, the module expands recursively along the first row
(Laplace/cofactor expansion):

$$
\det(A) = \sum_{j=1}^n (-1)^{1+j}\, a_{1j} \, \det(A_{1j})
$$

where $A_{1j}$ is $A$ with row 1 and column $j$ removed. A determinant of
zero is a warning sign: the transformation has *collapsed* space into a
lower dimension (e.g. squashed a plane onto a line), and information has
been lost forever.


In [16]:
square = Matrix([[4, 0], [0, 3]])
print("det =", square.det)   # scaling factor: 4 * 3 = 12

singular = Matrix([[1, 2], [2, 4]])   # second row is a multiple of the first - so it collapses to a line
print("det of a singular matrix =", singular.det)

det = 12.0
det of a singular matrix = 0.0


### The inverse — undoing the transformation :-

If $A$ represents a transformation, $A^{-1}$ is the transformation that
brings you back exactly where you started:

$$
A A^{-1} = A^{-1} A = I
$$

The module finds $A^{-1}$ with **Gauss-Jordan elimination**: augment $A$
with the identity matrix $[A \mid I]$, then row-reduce the left side down to
$I$ — whatever happens to the right side along the way *is* $A^{-1}$:

$$
[A \mid I] \;\xrightarrow{\text{row reduce}}\; [I \mid A^{-1}]
$$

A matrix can only be undone if it didn't collapse space in the first place
which is exactly why $\det(A) = 0$ makes a matrix **singular** (non-invertible).

In [17]:
inv = square.inverse()
print("A⁻¹ =\n", inv)
print("\ncheck A @ A⁻¹ = I:\n", square @ inv)

try:
    singular.inverse()
except ValueError as e:
    print("\nAs expected, the singular matrix refuses to invert :-", e)

A⁻¹ =
 Matrix([
 Vector([0.25, 0.0]),
 Vector([0.0, 0.3333333333333333])
])

check A @ A⁻¹ = I:
 Matrix([
 Vector([1.0, 0.0]),
 Vector([0.0, 1.0])
])

As expected, the singular matrix refuses to invert :- Matrix is singular (determinant is 0) and cannot be inverted.


---
## Chapter 4 — QR Decomposition: building a perfectly perpendicular world

Every matrix $A$ (with independent columns) can be split into :-

$$
A = QR
$$

where:
- $Q$ has **orthonormal columns** (all perpendicular, all unit length) a
  "clean" coordinate system built from $A$'s columns
- $R$ is **upper triangular**, it records how the original columns of $A$
  were built out of $Q$'s clean directions

### Gram-Schmidt: the recipe for $Q$

Take each column $\mathbf{a}_j$ of $A$ in turn. Subtract away whatever part
of it already points along the directions you've built so far, what's left
is guaranteed to be perpendicular to all of them:

$$
\mathbf{u}_j = \mathbf{a}_j - \sum_{k<j} \big(\mathbf{a}_j \cdot \mathbf{q}_k\big)\, \mathbf{q}_k,
\qquad
\mathbf{q}_j = \frac{\mathbf{u}_j}{\lVert \mathbf{u}_j \rVert}
$$

This is precisely `Matrix.qr()`, one column at a time. QR is the quiet
workhorse that makes the next chapter, *eigendecomposition*  possible.

In [ ]:
M = Matrix([[1, 1, 0],
            [1, 0, 1],
            [0, 1, 1]])

Q, R = M.qr()
print("Q (orthonormal columns) =\n", Q)
print("\nR (upper triangular) =\n", R)

# Q's columns should be mutually orthogonal and have unit length

q0, q1 = Q.get_column(0), Q.get_column(1)
print("\nq0 · q1 ≈ 0 ?", round(q0.dot(q1), 12))
print("‖q0‖ ≈ 1 ?", round(q0.norm(), 12))

# QR should reconstruct the original matrix

print("\nQ @ R reconstructs M:\n", Q @ R)

---
## Chapter 5 — Eigenvalues & Eigenvectors: directions that refuse to bend

Most vectors get knocked off their original direction when a matrix
transforms them. An **eigenvector** is special: the transformation only
*stretches or shrinks* it, it never rotates off its own line.

$$
A \mathbf{x} = \lambda \mathbf{x}
$$

- $\mathbf{x}$ -> the eigenvector (the direction that survives)
- $\lambda$ -> the eigenvalue (how much it's stretched, or flipped if negative)

### Finding them: the QR algorithm

There's no simple formula for eigenvalues beyond small matrices, so the
module uses an elegant iterative trick. Repeatedly QR-decompose the matrix
and multiply the factors back together in reverse order:

$$
A_0 = A, \qquad A_{k+1} = R_k Q_k \quad \text{where } A_k = Q_k R_k
$$

Like magic, $A_k$ converges to an (almost) diagonal matrix whose diagonal
entries **are** the eigenvalues. The module adds a **Wilkinson shift**, a smarter guess at each step, borrowed from the trailing $2\times 2$ block which makes the iteration converge dramatically faster.

In [19]:
A = Matrix([[2, 1],
            [1, 2]])

eigenvalues, eigenvectors = A.eig()
print("eigenvalues:", eigenvalues)

for val, vec in zip(eigenvalues, eigenvectors):
    print(f"\nλ = {val:.4f},  x = {vec}")
    # verifying A x ≈ λ x
    Ax = A @ Matrix([vec.data]).transpose
    lambda_x = vec * val
    print("  A·x  =", [round(row[0], 6) for row in Ax.rows])
    print("  λ·x  =", [round(c, 6) for c in lambda_x])


eigenvalues: [2.9999999999999996, 1.0]

λ = 3.0000,  x = Vector([0.7071067811865475, 0.7071067811865475])
  A·x  = [2.12132, 2.12132]
  λ·x  = [2.12132, 2.12132]

λ = 1.0000,  x = Vector([0.707106781186546, -0.707106781186549])
  A·x  = [0.707107, -0.707107]
  λ·x  = [0.707107, -0.707107]


---
## Chapter 6 — SVD,PCA & Pseudoinverse: where the whole story converges

### Singular Value Decomposition (SVD) :-

Eigendecomposition only fully works for square matrices. **SVD** generalises
the same idea of *"find the special directions and the stretch factors along
them"* to matrices of **any** shape:

$$
A = U \Sigma V^{T}
$$

- $V$ - orthonormal directions in the *input* space
- $U$ - orthonormal directions in the *output* space
- $\Sigma$ - the singular values: how much stretching happens along each
  matched pair of directions

The module builds this from what you already met: it eigendecomposes
$A^{T}A$ (always square!) to get $V$ and the singular values
$\sigma_i = \sqrt{\lambda_i}$, then derives $U$ from
$\mathbf{u}_i = \frac{1}{\sigma_i} A\mathbf{v}_i$.

In [20]:
M = Matrix([[3, 1],
            [1, 3],
            [0, 0]])

U, sigma, Vt = svd(M)
print("singular values:", [round(s, 4) for s in sigma])
print("\nU (3×3):\n", U)
print("\nVᵀ (2×2):\n", Vt)

singular values: [4.0, 2.0]

U (3×3):
 Matrix([
 Vector([0.7071067811865476, 0.707106781186546, 1.2086469832255486e-14]),
 Vector([0.7071067811865476, -0.7071067811865489, 8.53162576394505e-15]),
 Vector([0.0, 0.0, -1.0])
])

Vᵀ (2×2):
 Matrix([
 Vector([0.7071067811865476, 0.7071067811865476]),
 Vector([0.7071067811865468, -0.7071067811865481])
])


### Principal Component Analysis (PCA), SVD applied to data :-

Given a dataset, PCA asks: *"along which direction does the data spread out
the most?"* That direction is where the **principal components** are found by
centering the data (subtracting the mean) and running SVD on it:

$$
X_c = X - \bar{X}, \qquad X_c = U\Sigma V^T
$$

The columns of $V$ (rows of $V^T$) are the principal directions, and the
fraction of variance each one explains is:

$$
\text{explained variance ratio}_i = \frac{\sigma_i^2}{\sum_k \sigma_k^2}
$$

This is how the messy scatter of real-world data gets compressed down to
the handful of directions that actually matter in the same idea from
Chapter 1 (dot products, orthogonality) all the way through Chapter 5
(eigenvalues), now finishing the real work.

In [ ]:
# a small 2D dataset with an obvious diagonal trend
X = Matrix([
    [2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
    [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9],
])

components, explained_variance, transformed = pca(X, n_components=1)

print("principal direction:", components)
print("explained variance ratio:", [round(v, 4) for v in explained_variance])
print("\ndata projected onto that single direction (first 5 points):")
for row in transformed.rows[:5]:
    print(" ", row)

principal direction: Matrix([
 Vector([-0.6778733985280115, -0.735178655544408])
])
explained variance ratio: [0.9632]

data projected onto that single direction (first 5 points):
  Vector([-0.8279701862010876])
  Vector([1.777580325280429])
  Vector([-0.9921974944148884])
  Vector([-0.2742104159753993])
  Vector([-1.6758014186445394])


---
### The Moore-Penrose Pseudoinverse: inverting the un-invertible :-
 
Only square, full-rank matrices have a true inverse $A^{-1}$. But what about
a matrix that's rectangular, or singular? The **pseudoinverse** $A^{+}$
generalises the idea of "undoing" a transformation to *any* matrix in the
best possible sense, it gives the least-squares solution when no exact
inverse exists.
 
Once you have the SVD, $A^{+}$ falls right out of it:
 
$$
A = U \Sigma V^{T}
\qquad \Longrightarrow \qquad
A^{+} = V \Sigma^{+} U^{T}
$$
 
where $\Sigma^{+}$ is formed by transposing $\Sigma$ and taking the
reciprocal of every *nonzero* singular value:
 
$$
\Sigma^{+}_{ii} =
\begin{cases}
1/\sigma_i & \sigma_i > 0 \\
0 & \sigma_i = 0
\end{cases}
$$
 
Flipping the zeros to zero (rather than blowing up to infinity) is exactly
what makes this work even when $A$ is singular or non-square.
 
The module builds `pinv(A)` directly from the `svd()` function already in
this story, no new boilerplate just $\Sigma^{+}$ plugged into the
formula above.
 
$A^{+}$ is the key to solving least-squares problems
$A\mathbf{x} = \mathbf{b}$ when $A$ has no exact inverse, the bread and
butter of linear regression:
 
$$
\mathbf{x}^{*} = A^{+}\mathbf{b}
\quad \text{minimises} \quad
\lVert A\mathbf{x} - \mathbf{b} \rVert_2
$$

In [22]:
# non-square matrix (3x2) has no ordinary inverse 

A = Matrix([[1, 2],
            [3, 4],
            [5, 6]])   
 
P = pinv(A)
print("A (3*2) =\n", A)
print("\nA⁺ (2*3) =\n", P)
 
# for a full column-rank matrix, A⁺A = I (but AA⁺ ≠ I in general)

identity_check = P @ A
print("\nA⁺ @ A ≈ I :\n", identity_check)

A (3*2) =
 Matrix([
 Vector([1.0, 2.0]),
 Vector([3.0, 4.0]),
 Vector([5.0, 6.0])
])

A⁺ (2*3) =
 Matrix([
 Vector([-1.3333333333332669, -0.3333333333333147, 0.6666666666666364]),
 Vector([1.0833333333332809, 0.3333333333333187, -0.4166666666666427])
])

A⁺ @ A ≈ I :
 Matrix([
 Vector([0.999999999999971, 2.6201263381153694e-14]),
 Vector([2.3425705819590803e-14, 0.9999999999999805])
])
